Basic Data Cleaning flow(Steps)

1) Check Table Name 
2) Check Column Names
3) Check Data Types
4) Check Spelling Mistakes
5) Check Null values
6) Check Negative values
7) Check Unique values in Column
8) Adding New Columns
9) drop Duplicates

Packages

In [51]:
import pandas as pd
import datetime as dt 
import numpy as np

----> Reading Data

In [52]:
data=pd.read_csv("ChargingRecords.csv")
data.head()

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54,2022-09-15 23:59,185,20.36
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01,2022-09-14 21:31,90,10.19
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54,2022-09-14 19:54,60,6.78
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32,2022-09-30 0:16,344,37.65
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30,2022-09-26 0:30,300,33.81


----> Changing Data Types Of Columns

In [53]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72856 entries, 0 to 72855
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   UserID          72856 non-null  int64  
 1   ChargerID       72856 non-null  int64  
 2   ChargerCompany  72856 non-null  int64  
 3   Location        72856 non-null  object 
 4   ChargerType     72856 non-null  int64  
 5   StartDay        72856 non-null  object 
 6   StartTime       72856 non-null  object 
 7   EndDay          72856 non-null  object 
 8   EndTime         72856 non-null  object 
 9   StartDatetime   72856 non-null  object 
 10  EndDatetime     72856 non-null  object 
 11  Duration        72856 non-null  int64  
 12  Demand          72856 non-null  float64
dtypes: float64(1), int64(5), object(7)
memory usage: 7.2+ MB


In [54]:
data['StartDatetime'] = pd.to_datetime(
    data['StartDay'].astype(str) + ' ' +
    data['StartTime'].astype(str).str[-8:],
    errors='coerce'
)

data['EndDatetime'] = pd.to_datetime(
    data['EndDay'].astype(str) + ' ' +
    data['EndTime'].astype(str).str[-8:],
    errors='coerce'
)



----> Unique Data in Every columns

In [55]:
for i in data.columns:
    print(i,'--->',data[i].unique())

UserID ---> [   0   14  156 ... 2335 2128 2155]
ChargerID ---> [   1    2    3 ... 2649 2670 2671]
ChargerCompany ---> [1 0]
Location ---> ['hotel' 'company' 'apartment' 'resort' 'public area' 'public institution'
 'restaurant' 'accommodation' 'golf' 'sightseeing' 'market' 'camping'
 'public parking lot' 'bus garage']
ChargerType ---> [0 1]
StartDay ---> ['2022-09-15' '2022-09-14' '2022-09-29' '2022-09-25' '2022-09-24'
 '2022-08-11' '2022-08-10' '2022-08-09' '2022-08-07' '2022-08-06'
 '2022-08-05' '2022-08-23' '2022-08-22' '2022-08-19' '2022-08-18'
 '2022-08-17' '2022-07-02' '2022-07-03' '2022-07-05' '2022-07-08'
 '2022-07-14' '2022-07-27' '2022-06-07' '2022-06-09' '2022-06-10'
 '2022-06-18' '2022-06-23' '2022-05-08' '2022-05-09' '2022-05-12'
 '2022-05-14' '2022-05-15' '2022-05-16' '2022-05-23' '2022-05-27'
 '2022-04-08' '2022-04-14' '2022-04-16' '2022-04-27' '2022-03-01'
 '2022-03-02' '2022-03-03' '2022-03-10' '2022-03-12' '2022-03-13'
 '2022-03-18' '2022-03-30' '2022-03-31' '2022-02-

----> Null values in Columns

In [56]:
Null_Values=data.isnull().sum()
print(Null_Values)

UserID            0
ChargerID         0
ChargerCompany    0
Location          0
ChargerType       0
StartDay          0
StartTime         0
EndDay            0
EndTime           0
StartDatetime     0
EndDatetime       0
Duration          0
Demand            0
dtype: int64


----> Creating New Columns Based on Condeations

In [57]:
start_time_parsed = pd.to_datetime(
    data['StartTime'].astype(str).str[-8:],
    format='%H:%M:%S',
    errors='coerce'
)
data['StartHour']=start_time_parsed.dt.hour

hourly_summary = (
    data.groupby('StartHour')
    .agg(
        Total_Energy_kWh=('Demand', 'sum'),
        Charging_Sessions=('Demand', 'count')
    )
    .reset_index()
)

hourly_summary = hourly_summary.sort_values(
    'Total_Energy_kWh', ascending=False
)

print(hourly_summary.head(10))

    StartHour  Total_Energy_kWh  Charging_Sessions
18         18         119495.53               6512
19         19         110110.90               5505
17         17         108031.62               6357
20         20         102583.85               4761
21         21          90909.38               3979
16         16          80054.74               5146
8           8          72574.42               4143
15         15          59269.10               4215
9           9          58474.12               4058
22         22          57921.22               2434


In [58]:
low_energy_hours = data['Demand'].quantile(0.33)
high_energy_hours = data['Demand'].quantile(0.67)

data['DemandLevel']=np.select(
    [
        data['Demand'] <= low_energy_hours,
        data['Demand'] <= high_energy_hours,
    ],[
        
        'Low Demand',
        'Medium Demand',
    ],
    default='High Demand'
)

In [59]:
data['CalculatedDuration']=data['EndDatetime']-data['StartDatetime']

data['calculated_duration_minutes'] = (
    data['CalculatedDuration'].dt.total_seconds() / 60
)

# Check the results
print(data[
    [
        'StartDatetime',
        'EndDatetime',
        'CalculatedDuration',
        'calculated_duration_minutes'
    ]
].head(3))


        StartDatetime         EndDatetime CalculatedDuration  \
0 2022-09-15 20:54:02 2022-09-15 23:59:13    0 days 03:05:11   
1 2022-09-14 20:01:05 2022-09-14 21:31:04    0 days 01:29:59   
2 2022-09-14 18:54:30 2022-09-14 19:54:29    0 days 00:59:59   

   calculated_duration_minutes  
0                   185.183333  
1                    89.983333  
2                    59.983333  


In [60]:
data['DurationDifference'] = (
    data['calculated_duration_minutes'] - data['Duration']
)

print("Duration column data type:", data['Duration'].dtype)

print(
    data[
        ['Duration', 'calculated_duration_minutes', 'DurationDifference']
    ].head(10)
)

Duration column data type: int64
   Duration  calculated_duration_minutes  DurationDifference
0       185                   185.183333            0.183333
1        90                    89.983333           -0.016667
2        60                    59.983333           -0.016667
3       344                   343.850000           -0.150000
4       300                   299.983333           -0.016667
5        70                    69.983333           -0.016667
6        60                    59.983333           -0.016667
7       494                   494.033333            0.033333
8        60                    59.983333           -0.016667
9       112                   112.266667            0.266667


In [61]:
# Calculate exact elapsed seconds from the timestamps
data['CalculatedDurationSeconds'] = (
    data['EndDatetime'] - data['StartDatetime']
).dt.total_seconds()

# Convert the recorded duration into seconds
data['RecordedDurationSeconds'] = data['Duration'] * 60

# Check whether the recorded duration matches rounded elapsed minutes
data['RoundedDurationMatches'] = (
    data['Duration']
    == (data['CalculatedDurationSeconds'] / 60).round().astype('int64')
)

print("Exact elapsed seconds vs recorded seconds:")
print(
    data[
        [
            'Duration',
            'CalculatedDurationSeconds',
            'RecordedDurationSeconds',
            'RoundedDurationMatches'
        ]
    ].head(5)
)

print("\nRounded duration match counts:")
print(data['RoundedDurationMatches'].value_counts())

Exact elapsed seconds vs recorded seconds:
   Duration  CalculatedDurationSeconds  RecordedDurationSeconds  \
0       185                    11111.0                    11100   
1        90                     5399.0                     5400   
2        60                     3599.0                     3600   
3       344                    20631.0                    20640   
4       300                    17999.0                    18000   

   RoundedDurationMatches  
0                    True  
1                    True  
2                    True  
3                    True  
4                    True  

Rounded duration match counts:
RoundedDurationMatches
True     67180
False     5676
Name: count, dtype: int64


In [62]:
mismatches = data.loc[
    ~data['RoundedDurationMatches'],
    [
        'StartDatetime',
        'EndDatetime',
        'Duration',
        'CalculatedDurationSeconds',
        'RecordedDurationSeconds',
        'DurationDifference'
    ]
].copy()

print("Total mismatches:", len(mismatches))

print(mismatches.head(5).to_string(index=False))

mismatches['DifferenceMinutes'] = (
    mismatches['CalculatedDurationSeconds'] / 60
    - mismatches['Duration']
)

print("\nDifference summary in minutes:")

Total mismatches: 5676
      StartDatetime         EndDatetime  Duration  CalculatedDurationSeconds  RecordedDurationSeconds  DurationDifference
2022-08-19 17:10:57 2022-08-19 19:10:57         4                     7200.0                      240              116.00
2022-03-01 12:45:35 2022-03-01 13:34:38         0                     2943.0                        0               49.05
2022-03-13 09:49:06 2022-03-13 10:41:42         0                     3156.0                        0               52.60
2022-02-11 08:45:14 2022-02-11 10:00:14         0                     4500.0                        0               75.00
2022-08-13 19:21:33 2022-08-13 19:34:03        13                      750.0                      780               -0.50

Difference summary in minutes:


In [63]:
data['ActualElapsedMinutes'] = (
    data['CalculatedDurationSeconds'] / 60
)

data['DurationDifference'] = (
    data['ActualElapsedMinutes'] - data['Duration']
)

# Categorize the differences for investigation
data['DurationIssue'] = np.select(
    [
        data['Duration'] == 0,
        data['DurationDifference'].abs() <= 0.5,
        data['DurationDifference'].abs() > 0.5
    ],
    [
        'Recorded duration is zero',
        'Difference is 30 seconds or less',
        'Difference exceeds 30 seconds'
    ],
    default='Other'
)

print(data['DurationIssue'].value_counts())

print("\nExamples where recorded duration is zero but elapsed time is positive:")
print(
    data.loc[
        (data['Duration'] == 0) &
        (data['ActualElapsedMinutes'] > 0),
        ['StartDatetime', 'EndDatetime', 'Duration', 'ActualElapsedMinutes']
    ].head(5).to_string(index=False)
)

DurationIssue
Difference is 30 seconds or less    67146
Difference exceeds 30 seconds        4992
Recorded duration is zero             718
Name: count, dtype: int64

Examples where recorded duration is zero but elapsed time is positive:
      StartDatetime         EndDatetime  Duration  ActualElapsedMinutes
2022-03-01 12:45:35 2022-03-01 13:34:38         0             49.050000
2022-03-13 09:49:06 2022-03-13 10:41:42         0             52.600000
2022-02-11 08:45:14 2022-02-11 10:00:14         0             75.000000
2022-06-09 14:40:16 2022-06-09 14:40:39         0              0.383333
2022-08-01 11:14:23 2022-08-01 11:14:42         0              0.316667


In [64]:
# Records with large duration differences and nonzero recorded duration
large_mismatches = data.loc[
    (data['DurationDifference'].abs() > 0.5) &
    (data['Duration'] > 0),
    [
        'StartDatetime',
        'EndDatetime',
        'Duration',
        'ActualElapsedMinutes',
        'DurationDifference'
    ]
].copy()

print("Large mismatches with nonzero duration:", len(large_mismatches))

print("\nFirst 20 examples:")
print(large_mismatches.head(5).to_string(index=False))

print("\nLargest positive differences:")
print(
    large_mismatches.nlargest(10, 'DurationDifference')
    .to_string(index=False)
)

print("\nLargest negative differences:")
print(
    large_mismatches.nsmallest(10, 'DurationDifference')
    .to_string(index=False)
)

Large mismatches with nonzero duration: 4992

First 20 examples:
      StartDatetime         EndDatetime  Duration  ActualElapsedMinutes  DurationDifference
2022-08-19 17:10:57 2022-08-19 19:10:57         4            120.000000          116.000000
2022-02-15 19:24:44 2022-02-16 04:24:13       540            539.483333           -0.516667
2022-04-20 20:10:52 2022-04-21 05:10:21       540            539.483333           -0.516667
2022-06-30 20:39:32 2022-07-01 06:02:28       557            562.933333            5.933333
2022-07-06 09:38:43 2022-07-06 10:34:26        55             55.716667            0.716667

Largest positive differences:
      StartDatetime         EndDatetime  Duration  ActualElapsedMinutes  DurationDifference
2022-05-14 13:59:33 2022-05-17 20:36:35       397           4717.033333         4320.033333
2021-10-01 18:31:08 2021-10-04 13:17:25       101           4006.283333         3905.283333
2022-03-30 22:04:34 2022-04-02 15:00:15       157           3895.683333     

In [65]:
# Inspect the recorded duration values
print("Recorded Duration statistics:")
print(data['Duration'].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nDuration values equal to zero:", (data['Duration'] == 0).sum())
print("Duration values above 24 hours:", (data['Duration'] > 1440).sum())

print("\nLongest sessions according to timestamps:")
print(
    data.assign(
        ActualElapsedMinutes=(
            data['EndDatetime'] - data['StartDatetime']
        ).dt.total_seconds() / 60
    )[
        ['StartDatetime', 'EndDatetime', 'Duration', 'ActualElapsedMinutes']
    ]
    .nlargest(10, 'ActualElapsedMinutes')
    .to_string(index=False)
)

Recorded Duration statistics:


count    72856.000000
mean       151.194054
std        146.430286
min        -29.000000
25%         40.000000
50%        112.000000
75%        202.000000
90%        352.000000
95%        458.000000
99%        634.000000
max       1573.000000
Name: Duration, dtype: float64

Duration values equal to zero: 718
Duration values above 24 hours: 1

Longest sessions according to timestamps:
      StartDatetime         EndDatetime  Duration  ActualElapsedMinutes
2022-05-14 13:59:33 2022-05-17 20:36:35       397           4717.033333
2021-10-01 18:31:08 2021-10-04 13:17:25       101           4006.283333
2022-03-30 22:04:34 2022-04-02 15:00:15       157           3895.683333
2022-03-24 22:08:20 2022-03-27 13:01:11       893           3772.850000
2022-03-05 09:09:11 2022-03-07 12:09:13       132           3060.033333
2021-10-25 08:57:16 2021-10-27 09:00:02         3           2882.766667
2021-10-23 23:54:59 2021-10-25 23:50:59       167           2876.000000
2022-01-21 23:14:22 2022-01-23 22:28:5

In [66]:
# 1. Negative recorded durations
negative_duration = data[data['Duration'] < 0]

print("Negative Duration records:", len(negative_duration))
print(
    negative_duration[
        ['StartDatetime', 'EndDatetime', 'Duration']
    ].head(5).to_string(index=False)
)

Negative Duration records: 17
      StartDatetime         EndDatetime  Duration
2022-02-15 15:04:54 2022-02-15 14:35:36       -29
2022-02-24 14:27:44 2022-02-24 14:18:35        -9
2022-01-17 14:47:03 2022-01-17 14:39:24        -8
2022-04-29 18:58:46 2022-04-29 18:52:20        -6
2022-05-07 16:24:20 2022-05-07 16:03:03       -21


In [67]:
# 2. End timestamps earlier than start timestamps
invalid_timestamps = data[
    data['EndDatetime'] < data['StartDatetime']
]

print("\nEnd before start records:", len(invalid_timestamps))
print(
    invalid_timestamps[
        ['StartDatetime', 'EndDatetime', 'Duration']
    ].head(5).to_string(index=False)
)



End before start records: 17
      StartDatetime         EndDatetime  Duration
2022-02-15 15:04:54 2022-02-15 14:35:36       -29
2022-02-24 14:27:44 2022-02-24 14:18:35        -9
2022-01-17 14:47:03 2022-01-17 14:39:24        -8
2022-04-29 18:58:46 2022-04-29 18:52:20        -6
2022-05-07 16:24:20 2022-05-07 16:03:03       -21


In [68]:
# 3. Sessions spanning more than 24 hours
data['ActualElapsedHours'] = (
    data['EndDatetime'] - data['StartDatetime']
).dt.total_seconds() / 3600

long_sessions = data[data['ActualElapsedHours'] > 24]

print("\nSessions spanning over 24 hours:", len(long_sessions))
print(
    long_sessions[
        ['StartDatetime', 'EndDatetime', 'Duration', 'ActualElapsedHours']
    ].head(10).to_string(index=False)
)


Sessions spanning over 24 hours: 21
      StartDatetime         EndDatetime  Duration  ActualElapsedHours
2022-06-18 16:44:29 2022-06-19 16:49:20       472           24.080833
2022-03-08 18:24:41 2022-03-10 12:49:59       229           42.421667
2021-10-16 17:51:50 2021-10-18 09:15:16       175           39.390556
2021-10-23 23:54:59 2021-10-25 23:50:59       167           47.933333
2022-03-07 12:33:55 2022-03-08 17:25:38       147           28.861944
2022-03-05 09:09:11 2022-03-07 12:09:13       132           51.000556
2022-02-15 11:07:51 2022-02-17 04:15:59       181           41.135556
2022-03-30 22:04:34 2022-04-02 15:00:15       157           64.928056
2021-10-01 18:31:08 2021-10-04 13:17:25       101           66.771389
2022-01-21 23:14:22 2022-01-23 22:28:59       214           47.243611


In [69]:
# Create independent data-quality checks
quality_summary = {
    'Total records': len(data),
    'Negative recorded durations': int((data['Duration'] < 0).sum()),
    'End before start': int((data['EndDatetime'] < data['StartDatetime']).sum()),
    'Zero recorded durations': int((data['Duration'] == 0).sum()),
    'Sessions over 24 hours': int((data['ActualElapsedHours'] > 24).sum()),
    'Duration mismatch over 30 seconds': int(
        (data['DurationDifference'].abs() > 0.5).sum()
    ),
    'Missing start timestamps': int(data['StartDatetime'].isna().sum()),
    'Missing end timestamps': int(data['EndDatetime'].isna().sum())
}

for issue, count in quality_summary.items():
    print(f'{issue}: {count}')

Total records: 72856
Negative recorded durations: 17
End before start: 17
Zero recorded durations: 718
Sessions over 24 hours: 21
Duration mismatch over 30 seconds: 5168
Missing start timestamps: 0
Missing end timestamps: 0


In [70]:
data = data.drop(
    [
        'CalculatedDurationSeconds',
        'RecordedDurationSeconds',
        'RoundedDurationMatches',
        'DurationIssue',
        'ActualElapsedHours', 
        'ActualElapsedMinutes',
        'CalculatedDuration',
        'DurationDifference'
    ],
    axis=1,
    errors='ignore'
)

In [71]:
data.head(4)

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,StartHour,DemandLevel,calculated_duration_minutes
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,2022-09-15 23:59:13,185,20.36,20,High Demand,185.183333
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,2022-09-14 21:31:04,90,10.19,20,Medium Demand,89.983333
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:30,2022-09-14 19:54:29,60,6.78,18,Low Demand,59.983333
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:51,2022-09-30 00:16:42,344,37.65,18,High Demand,343.850000


In [72]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72856 entries, 0 to 72855
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   UserID                       72856 non-null  int64         
 1   ChargerID                    72856 non-null  int64         
 2   ChargerCompany               72856 non-null  int64         
 3   Location                     72856 non-null  object        
 4   ChargerType                  72856 non-null  int64         
 5   StartDay                     72856 non-null  object        
 6   StartTime                    72856 non-null  object        
 7   EndDay                       72856 non-null  object        
 8   EndTime                      72856 non-null  object        
 9   StartDatetime                72856 non-null  datetime64[ns]
 10  EndDatetime                  72856 non-null  datetime64[ns]
 11  Duration                     72856 non-nu

In [73]:
data['ChargerType'].value_counts()

ChargerType
0    57903
1    14953
Name: count, dtype: int64

In [74]:
data['ChargerSpeed']=np.where(data['ChargerType']==0,'SlowCharger','FastCharger')
data.head(1)

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,StartHour,DemandLevel,calculated_duration_minutes,ChargerSpeed
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,2022-09-15 23:59:13,185,20.36,20,High Demand,185.183333,SlowCharger


In [75]:
data['Time Taken for Charging']=(data['EndDatetime'] - data['StartDatetime'])  
data['charging Days']=data['Time Taken for Charging'].dt.days


In [76]:
data['charging Time']=(data['Time Taken for Charging']
                       .apply(lambda i: str(i).split(' days ')[-1])
                       .apply(lambda i: dt.datetime.strptime(i.replace('+','').strip(), '%H:%M:%S').time()))

In [77]:
data['charging Hours']=data['Time Taken for Charging'].dt.seconds//3600
data['charging Minutes']=(data['Time Taken for Charging'].dt.seconds//60)%60

In [78]:
data.head(2)

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,...,Demand,StartHour,DemandLevel,calculated_duration_minutes,ChargerSpeed,Time Taken for Charging,charging Days,charging Time,charging Hours,charging Minutes
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:02,...,20.36,20,High Demand,185.183333,SlowCharger,0 days 03:05:11,0,03:05:11,3,5
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:05,...,10.19,20,Medium Demand,89.983333,SlowCharger,0 days 01:29:59,0,01:29:59,1,29


In [79]:
data['ChargerCompany'].value_counts()

ChargerCompany
1    45713
0    27143
Name: count, dtype: int64

In [80]:
data['ChargerCompanyName']=data['ChargerCompany'].map({0:'Company A',1:'Company B'})

In [81]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72856 entries, 0 to 72855
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype          
---  ------                       --------------  -----          
 0   UserID                       72856 non-null  int64          
 1   ChargerID                    72856 non-null  int64          
 2   ChargerCompany               72856 non-null  int64          
 3   Location                     72856 non-null  object         
 4   ChargerType                  72856 non-null  int64          
 5   StartDay                     72856 non-null  object         
 6   StartTime                    72856 non-null  object         
 7   EndDay                       72856 non-null  object         
 8   EndTime                      72856 non-null  object         
 9   StartDatetime                72856 non-null  datetime64[ns] 
 10  EndDatetime                  72856 non-null  datetime64[ns] 
 11  Duration                    

----> Finding Negative Values From Every Columns

In [82]:
for i in data.columns:
    if data[i].dtype=='int' or data[i].dtype=='float':
        if data[i].min()< 0:
            print(i)


Duration
calculated_duration_minutes
charging Days


In [83]:
for i,j in zip(data['Duration'],data['charging Days']):
    if i<0:
        print(i,j)
        

-29 -1
-9 -1
-8 -1
-6 -1
-21 -1
-6 -1
-7 -1
-2 -1
-20 -1
-9 -1
-13 -1
-6 -1
-16 -1
-5 -1
-8 -1
-24 -1
-16 -1


In [84]:
bad_times=data.loc[data['EndDatetime'] < data['StartDatetime'],
                  ['UserID','StartDatetime','EndDatetime','Duration','charging Days']]

print(bad_times.to_string(index=False))

bad_times["Flag"] = "EndDatetime before StartDatetime"

 UserID       StartDatetime         EndDatetime  Duration  charging Days
   1403 2022-02-15 15:04:54 2022-02-15 14:35:36       -29             -1
   1403 2022-02-24 14:27:44 2022-02-24 14:18:35        -9             -1
   1403 2022-01-17 14:47:03 2022-01-17 14:39:24        -8             -1
     37 2022-04-29 18:58:46 2022-04-29 18:52:20        -6             -1
     55 2022-05-07 16:24:20 2022-05-07 16:03:03       -21             -1
     74 2022-04-29 15:48:51 2022-04-29 15:43:07        -6             -1
     92 2022-03-27 15:17:53 2022-03-27 15:10:38        -7             -1
     92 2022-02-05 18:52:39 2022-02-05 18:50:15        -2             -1
    266 2022-05-08 12:02:28 2022-05-08 11:42:02       -20             -1
    326 2022-01-09 14:29:15 2022-01-09 14:20:28        -9             -1
   1768 2022-04-07 12:25:32 2022-04-07 12:12:11       -13             -1
     55 2022-04-09 11:29:43 2022-04-09 11:23:46        -6             -1
   1474 2022-05-05 17:48:01 2022-05-05 17:31:39    

----> Checking & Removing Duplicates

In [85]:
data.duplicated().sum()

np.int64(13)

In [86]:
data=data.drop_duplicates()
data.duplicated().sum()

np.int64(0)

In [87]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72843 entries, 0 to 72855
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype          
---  ------                       --------------  -----          
 0   UserID                       72843 non-null  int64          
 1   ChargerID                    72843 non-null  int64          
 2   ChargerCompany               72843 non-null  int64          
 3   Location                     72843 non-null  object         
 4   ChargerType                  72843 non-null  int64          
 5   StartDay                     72843 non-null  object         
 6   StartTime                    72843 non-null  object         
 7   EndDay                       72843 non-null  object         
 8   EndTime                      72843 non-null  object         
 9   StartDatetime                72843 non-null  datetime64[ns] 
 10  EndDatetime                  72843 non-null  datetime64[ns] 
 11  Duration                     7284

----> Renameing Columns Names

In [88]:

data=data.rename(columns = {
    'UserID': 'user_id',
    'ChargerID': 'charger_id',
    'ChargerCompany': 'charger_company_code',
    'Location': 'charging_location',
    'ChargerType': 'charger_type_code',
    'StartDay': 'start_day',
    'StartTime': 'start_time',
    'EndDay': 'end_day',
    'EndTime': 'end_time',
    'StartDatetime': 'start_datetime',
    'EndDatetime': 'end_datetime',
    'Duration': 'duration_minutes',
    'Demand': 'demand_value',
    'StartHour': 'start_hour',
    'DemandLevel': 'demand_level',
    'CalculatedDuration': 'calculated_duration',
    'DurationDays': 'duration_days',
    'DurationHours': 'duration_hours',
    'DurationMinutes': 'duration_minutes',
    'CalculatedDurationMinutes': 'calculated_duration_minutes',
    'ChargerSpeed': 'charger_speed_category',
    'Time Taken for Charging': 'time_taken_for_charging',
    'charging Days': 'charging_days',
    'charging Time': 'charging_time',
    'charging Hours': 'charging_hours',
    'charging Minutes': 'charging_minutes',
    'ChargerCompanyName': 'charger_company_name'
}
)

print("Total columns:", len(data.columns))
print(data.columns.tolist())

Total columns: 23
['user_id', 'charger_id', 'charger_company_code', 'charging_location', 'charger_type_code', 'start_day', 'start_time', 'end_day', 'end_time', 'start_datetime', 'end_datetime', 'duration_minutes', 'demand_value', 'start_hour', 'demand_level', 'calculated_duration_minutes', 'charger_speed_category', 'time_taken_for_charging', 'charging_days', 'charging_time', 'charging_hours', 'charging_minutes', 'charger_company_name']


In [89]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72843 entries, 0 to 72855
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype          
---  ------                       --------------  -----          
 0   user_id                      72843 non-null  int64          
 1   charger_id                   72843 non-null  int64          
 2   charger_company_code         72843 non-null  int64          
 3   charging_location            72843 non-null  object         
 4   charger_type_code            72843 non-null  int64          
 5   start_day                    72843 non-null  object         
 6   start_time                   72843 non-null  object         
 7   end_day                      72843 non-null  object         
 8   end_time                     72843 non-null  object         
 9   start_datetime               72843 non-null  datetime64[ns] 
 10  end_datetime                 72843 non-null  datetime64[ns] 
 11  duration_minutes             7284

----> Creating new Cleaned_data Csv file With-Out bad_times

In [90]:
cleaned_data = data.loc[
    data["end_datetime"] >= data["start_datetime"]
].copy()

In [91]:
bad_times.to_csv("flagged_records.csv", index=False)

In [92]:
cleaned_data["start_datetime"] = pd.to_datetime(cleaned_data["start_datetime"])
cleaned_data["end_datetime"] = pd.to_datetime(cleaned_data["end_datetime"])

In [93]:
(cleaned_data["end_datetime"] < cleaned_data["start_datetime"]).sum()

np.int64(0)

In [94]:
cleaned_data.to_csv("cleaned_data.csv", index=False)

Cleaned_Data Set Validation 

----> Checking Any Negative Values & Duplicates & Null Values & DataTypes in cleaned_data set

In [95]:
for i in cleaned_data.columns:
    if pd.api.types.is_numeric_dtype(cleaned_data[i]):
        if cleaned_data[i].min() < 0:
            print(i)

In [96]:
cleaned_data.duplicated().sum()

np.int64(0)

In [97]:
cleaned_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72826 entries, 0 to 72855
Data columns (total 23 columns):
 #   Column                       Non-Null Count  Dtype          
---  ------                       --------------  -----          
 0   user_id                      72826 non-null  int64          
 1   charger_id                   72826 non-null  int64          
 2   charger_company_code         72826 non-null  int64          
 3   charging_location            72826 non-null  object         
 4   charger_type_code            72826 non-null  int64          
 5   start_day                    72826 non-null  object         
 6   start_time                   72826 non-null  object         
 7   end_day                      72826 non-null  object         
 8   end_time                     72826 non-null  object         
 9   start_datetime               72826 non-null  datetime64[ns] 
 10  end_datetime                 72826 non-null  datetime64[ns] 
 11  duration_minutes             7282

In [98]:
cleaned_data.isnull().sum()

user_id                        0
charger_id                     0
charger_company_code           0
charging_location              0
charger_type_code              0
start_day                      0
start_time                     0
end_day                        0
end_time                       0
start_datetime                 0
end_datetime                   0
duration_minutes               0
demand_value                   0
start_hour                     0
demand_level                   0
calculated_duration_minutes    0
charger_speed_category         0
time_taken_for_charging        0
charging_days                  0
charging_time                  0
charging_hours                 0
charging_minutes               0
charger_company_name           0
dtype: int64